# ConvNeXt V2 baseline on CheXpert — Colab

Trains ConvNeXt V2 (`facebook/convnextv2-base-22k-224`, ImageNet-22k pretrained, via Hugging Face) on CheXpert. Like RAD-DINO, ConvNeXt V2 needed a bit more than a config toggle: the pinned `timm==0.5.4` in this pipeline predates ConvNeXt V2 (it was only added to `timm` well after this repo's 2021-era dependency pins), so patch 11 in cell 7 loads it via `transformers` instead and wraps it with a plain linear head, the same pattern used for RAD-DINO.

**Before running:**
1. `Runtime > Change runtime type` → GPU.
2. ConvNeXt V2 is initialized from **ImageNet-22k pretrained weights**, auto-downloaded, no manual checkpoint needed.
3. **Mixed precision is disabled for this notebook** (`args.use_autocast = False` in cell 12) -- ConvNeXt V2's depthwise convolutions hit a cuDNN "no engine found" error under bf16 autocast (verified locally: fails under autocast, works fine in plain fp32). Patch 12 in cell 7 makes this toggleable rather than hardcoded, so every other notebook's behavior is unaffected.

Cell 12 has a `NUM_CLASSES` toggle: **14** (current default) trains the head on all 14 CheXpert labels but only reports on the 5 competition classes; **5** trains directly on just those 5.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# Colab's pre-installed stack (pandas, etc.) is compiled against numpy>=2's actual C ABI --
# this isn't just a pip-metadata preference, running them against numpy<2 crashes for real
# ("numpy.dtype size changed, may indicate binary incompatibility"). We previously tried
# downgrading numpy to dodge an *unrelated* numpy-2.0 incompatibility in imgaug (a transitive
# dep of albumentations) -- that broke pandas instead, and the downgrade didn't even take
# effect this-session anyway, since Colab pre-imports numpy before any of our cells run. So:
# make sure numpy is restored to >=2 here (undoing any earlier accidental downgrade left on
# disk), and fix the imgaug incompatibility directly instead (see section 2, patches 4 & 5).
import sys
!{sys.executable} -m pip install -q --break-system-packages "numpy>=2"
import numpy
print('numpy', numpy.__version__, '@', numpy.__file__)
print('kernel python:', sys.executable)

## 1. Kaggle credentials + dataset download

In [ ]:
import os, json, getpass, sys

kaggle_username = input('Kaggle username: ')
kaggle_key = getpass.getpass('Kaggle API key: ')

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
del kaggle_key

!{sys.executable} -m pip install -q --break-system-packages kaggle
os.makedirs('/content/data', exist_ok=True)

# Using the kaggle package's own Python API instead of any CLI form -- it turns out this
# version of the package has no `__main__.py` (`-m kaggle` fails with "cannot be directly
# executed"), and the bare `kaggle` console script hits the same PATH-mismatch class of
# issue we saw with pip earlier. Calling the API in-process sidesteps all of that.
import kaggle
kaggle.api.authenticate()
# quiet=False -- dataset_download_files defaults to quiet=True, which is why this ran with
# zero output before. With it off, kaggle's own tqdm byte-progress bar (MB downloaded, speed,
# ETA) shows up instead of the download appearing to hang silently for several minutes.
kaggle.api.dataset_download_files('ashery/chexpert', path='/content/data', unzip=True, quiet=False)
print('Download complete.')

In [ ]:
# The train.csv/valid.csv paths are prefixed with 'CheXpert-v1.0-small/', but the
# Kaggle extract puts train/ and valid/ directly under /content/data. Symlink to bridge it.
import os
data_root = '/content/data'
link = os.path.join(data_root, 'CheXpert-v1.0-small')
if not os.path.islink(link) and not os.path.exists(link):
    os.symlink(data_root, link)

print(sorted(os.listdir(data_root)))
import pandas as pd
df = pd.read_csv(os.path.join(data_root, 'train.csv'))
print(df.shape, list(df.columns))

## 2. Clone the CheXpert finetuning codebase, patch it, and install dependencies
Reuses an existing open-source CheXpert finetuning pipeline (dataloader, training loop, metrics) -- not any pretrained model from it, just its well-tested training code. Same 2021-era dependency pins as the other baseline notebooks in this project (`timm==0.5.4`, `albumentations==0.4.3`; `transformers` is also already in the install list, needed for ConvNeXt V2), same patches:

- `np.bool` alias fix, dead import removal (patches 1-2)
- `class_indices` support for training on just the 5 competition classes (patch 3)
- Real bf16 `torch.autocast` mixed precision (patch 6, but see patch 12 below -- disabled for this notebook specifically)
- `meanRecall`/`meanAccuracy` metrics, threaded through both reporting paths (patches 7-8)
- Clean per-epoch results CSV (patch 9) and per-epoch weights-only checkpoints (patch 10)

Plus two new patches specific to this notebook:
- **Patch 11**: adds a `ConvNextV2Classifier` wrapper (ConvNeXt V2's encoder, loaded via `transformers`, plus a linear head) and wires it in as `model_name="convnextv2_base"`, since the model-selector has no support for non-`timm` models by default and `timm==0.5.4` predates ConvNeXt V2 entirely.
- **Patch 12**: makes patch 6's bf16 autocast conditionally toggleable via `args.use_autocast` (default `True`, so every other notebook is unaffected) instead of hardcoded on -- needed because ConvNeXt V2's depthwise convolutions hit a cuDNN error under bf16 autocast on at least some GPU/cuDNN combinations.

This cell is safe to re-run (e.g. after a kernel restart) -- the clone and patches are skipped/no-op if already present.

In [ ]:
import os, sys
# Clones the working CheXpert training code (dataloader/trainer/engine/model-selector) from
# its open-source repo, then extracts just that code into a plainly-named local folder --
# nothing here loads any pretrained foundation-model weights from that codebase, this is
# purely reusing the training pipeline itself.
if not os.path.exists('/content/chexpert_code'):
    !git clone -q https://github.com/jlianglab/Ark.git /content/_code_src
    os.rename('/content/_code_src/Ark_Plus/Finetuning', '/content/chexpert_code')
    import shutil; shutil.rmtree('/content/_code_src', ignore_errors=True)
%cd /content/chexpert_code

# Patch 1: np.bool removed in numpy>=1.24 (numpy-version-agnostic fix, not a numpy downgrade)
!sed -i 's/astype(np.bool)/astype(bool)/g' utils.py

# Patch 2: models.py imports UperNet_swin from a module not present in this repo folder,
# and never actually uses it -- dead import, crashes on load otherwise.
!sed -i '/from upernet_swin_transformer import UperNet_swin/d' models.py

# Patch 3: let CheXpert(...) select an arbitrary subset/order of label columns via class_indices,
# so we can train the model head on exactly the 5 competition classes instead of all 14.
# Idempotent: skips cleanly if already patched (e.g. re-running after a kernel restart).
_p = 'dataloader.py'
_s = open(_p).read()
_old = '''  def __init__(self, images_path, file_path, augment, num_class=14,
               uncertain_label="LSR-Ones", unknown_label=0, few_shot = -1):

    self.img_list = []
    self.img_label = []
    self.augment = augment
    assert uncertain_label in ["Ones", "Zeros", "LSR-Ones", "LSR-Zeros"]
    self.uncertain_label = uncertain_label

    with open(file_path, "r") as fileDescriptor:
      csvReader = csv.reader(fileDescriptor)
      next(csvReader, None)
      for line in csvReader:
        imagePath = os.path.join(images_path, line[0])
        if "test" in line[0]:
          label = line[1:]
        else:
          label = line[5:]
        for i in range(num_class):'''
_new = '''  def __init__(self, images_path, file_path, augment, num_class=14,
               uncertain_label="LSR-Ones", unknown_label=0, few_shot = -1, class_indices=None):

    self.img_list = []
    self.img_label = []
    self.augment = augment
    assert uncertain_label in ["Ones", "Zeros", "LSR-Ones", "LSR-Zeros"]
    self.uncertain_label = uncertain_label

    with open(file_path, "r") as fileDescriptor:
      csvReader = csv.reader(fileDescriptor)
      next(csvReader, None)
      for line in csvReader:
        imagePath = os.path.join(images_path, line[0])
        if class_indices is not None:
          label = [line[i] for i in class_indices]
          num_class = len(class_indices)
        elif "test" in line[0]:
          label = line[1:]
        else:
          label = line[5:]
        for i in range(num_class):'''
if _new in _s:
    print('dataloader.py already patched, skipping.')
elif _old in _s:
    open(_p, 'w').write(_s.replace(_old, _new))
    print('Patch applied.')
else:
    raise AssertionError("dataloader.py CheXpert.__init__ source did not match either expected pattern -- repo may have changed upstream")

!{sys.executable} -m pip install -q --break-system-packages timm==0.5.4 albumentations==0.4.3 opencv-python-headless scikit-image scikit-learn \
    simpleitk scipy pydicom yacs einops transformers pretrainedmodels imgaug pillow tqdm

# Patch 4 & 5: imgaug (a transitive dep of albumentations==0.4.3, old and unmaintained) has a
# np.sctypes reference (removed in numpy 2.0) in parameters.py, plus a dead `from scipy import
# misc` import (removed in modern scipy) in the same file. Both crash on *import*, so we can't
# `import imgaug` first to locate the file -- that import is exactly what crashes. Using
# importlib.util.find_spec instead, which locates the module without executing it.
import importlib.util, os as _os
_ia_dir = _os.path.dirname(importlib.util.find_spec('imgaug').origin)
_ia_params = _os.path.join(_ia_dir, 'parameters.py')
_s = open(_ia_params).read()
_before = _s
_s = _s.replace('np.sctypes["float"]', '[np.float16, np.float32, np.float64]')
_s = _s.replace('from scipy import misc\n', '')
if _s != _before:
    open(_ia_params, 'w').write(_s)
    print('imgaug patched:', _ia_params)
else:
    print('imgaug already patched (or patterns not found), skipping:', _ia_params)

# Now safe to actually import it and confirm.
import imgaug
print('imgaug import OK, version', imgaug.__version__)

# Patch 6: trainer.py creates a NativeScaler but never actually wraps the forward pass in
# torch.autocast, so training runs in plain fp32 -- no mixed-precision speedup despite
# appearances. Adding real autocast (bf16 -- native on A100, no gradient-scaling needed
# unlike fp16) around the forward+loss in train_one_epoch/evaluate, and around the forward
# pass in test_classification/test_model (also run every epoch via test_every_epoch=True).
# Also makes train_one_epoch return its average loss (previously computed but discarded) --
# needed for the clean per-epoch results table built in patch 9.
# Verified locally: same correctness, real speedup expected on Ampere+ GPUs.
_p = 'trainer.py'
_s = open(_p).read()

_old1 = '''  for i, (samples, targets) in enumerate(data_loader_train):
    samples, targets = samples.float().to(device), targets.float().to(device)
    
    outputs = model(samples)
    #outputs = torch.sigmoid(outputs)
    loss = criterion(outputs, targets)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.update(loss.item(), samples.size(0))
    batch_time.update(time.time() - end)
    end = time.time()

    if i % 50 == 0:
      progress.display(i)'''
_new1 = '''  for i, (samples, targets) in enumerate(data_loader_train):
    samples, targets = samples.float().to(device), targets.float().to(device)

    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
      outputs = model(samples)
      #outputs = torch.sigmoid(outputs)
      loss = criterion(outputs, targets)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.update(loss.item(), samples.size(0))
    batch_time.update(time.time() - end)
    end = time.time()

    if i % 50 == 0:
      progress.display(i)

  return losses.avg'''

_old2 = '''    for i, (samples, targets) in enumerate(data_loader_val):
      samples, targets = samples.float().to(device), targets.float().to(device)

      outputs = model(samples)
      #outputs = torch.sigmoid(outputs)
      loss = criterion(outputs, targets)'''
_new2 = '''    for i, (samples, targets) in enumerate(data_loader_val):
      samples, targets = samples.float().to(device), targets.float().to(device)

      with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        outputs = model(samples)
        #outputs = torch.sigmoid(outputs)
        loss = criterion(outputs, targets)'''

_old3 = '''      varInput = torch.autograd.Variable(samples.view(-1, c, h, w).cuda())

      out = model(varInput)'''
_new3 = '''      varInput = torch.autograd.Variable(samples.view(-1, c, h, w).cuda())

      with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        out = model(varInput)'''

if _new1 in _s and _new2 in _s and _s.count(_new3) == 2:
    print('trainer.py already patched, skipping.')
else:
    assert _old1 in _s, "trainer.py train_one_epoch pattern not found -- repo may have changed upstream"
    assert _old2 in _s, "trainer.py evaluate pattern not found -- repo may have changed upstream"
    assert _s.count(_old3) == 2, "trainer.py test_classification/test_model pattern not found (expected 2 occurrences) -- repo may have changed upstream"
    _s = _s.replace(_old1, _new1).replace(_old2, _new2).replace(_old3, _new3)
    open(_p, 'w').write(_s)
    print('trainer.py patched with autocast + train_loss return.')

# Patch 7: add meanRecall/meanAccuracy to utils.py, alongside the existing meanAUC/meanMCC/
# meanAP/meanF1 (same per-class Youden's-J-optimal-threshold binarization approach). Neither
# recall nor accuracy is computed anywhere in this repo for the CheXpert dataset otherwise --
# accuracy_score is only ever invoked for RSNAPneumonia/COVIDx.
_p = 'utils.py'
_s = open(_p).read()
_old_import = 'from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, average_precision_score, f1_score, matthews_corrcoef'
_new_import = 'from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, average_precision_score, f1_score, matthews_corrcoef, recall_score'
_old_anchor = '''    mf1_score = np.mean(f1_scores)
    return mf1_score, f1_scores

def metric_AUROC(target, output, nb_classes=14):'''
_new_anchor = '''    mf1_score = np.mean(f1_scores)
    return mf1_score, f1_scores

def meanRecall(ground_truth, predictions):
    # Compute mean Recall, using the same per-class Youden's-J-optimal threshold as meanF1/meanMCC
    recall_scores = []
    for i in range(ground_truth.shape[1]):
        if np.any(ground_truth[:, i]):
            fpr, tpr, thresholds = roc_curve(ground_truth[:, i], predictions[:, i])
            youden_j = tpr - fpr
            optimal_threshold = thresholds[np.argmax(youden_j)]
            binary_predictions = (predictions[:, i] > optimal_threshold).astype(int)
            recall = recall_score(ground_truth[:, i], binary_predictions)
            recall_scores.append(recall)

    mrecall_score = np.mean(recall_scores)
    return mrecall_score, recall_scores

def meanAccuracy(ground_truth, predictions):
    # Compute mean Accuracy, using the same per-class Youden's-J-optimal threshold as meanF1/meanMCC
    accuracy_scores = []
    for i in range(ground_truth.shape[1]):
        if np.any(ground_truth[:, i]):
            fpr, tpr, thresholds = roc_curve(ground_truth[:, i], predictions[:, i])
            youden_j = tpr - fpr
            optimal_threshold = thresholds[np.argmax(youden_j)]
            binary_predictions = (predictions[:, i] > optimal_threshold).astype(int)
            acc = accuracy_score(ground_truth[:, i], binary_predictions)
            accuracy_scores.append(acc)

    maccuracy_score = np.mean(accuracy_scores)
    return maccuracy_score, accuracy_scores

def metric_AUROC(target, output, nb_classes=14):'''
if 'def meanRecall' in _s:
    print('utils.py already patched, skipping.')
else:
    assert _old_import in _s, "utils.py sklearn import line not found -- repo may have changed upstream"
    assert _old_anchor in _s, "utils.py meanF1/metric_AUROC anchor not found -- repo may have changed upstream"
    _s = _s.replace(_old_import, _new_import).replace(_old_anchor, _new_anchor)
    open(_p, 'w').write(_s)
    print('utils.py patched with meanRecall/meanAccuracy.')

# Patch 8: thread mRecall/mAccuracy through engine.py's two reporting paths (per-epoch, and
# the final all-trials summary) alongside the existing mAUC/mMCC/mAP/mF1. Also fixes a
# pre-existing bug in the original repo's final-summary writer.write call, which mislabeled
# f1_scores as "AP =" and auc_scores as "F1 =" (the print() statements right above it were
# already correct -- only the file write had the swapped variables). Verified locally
# end-to-end (both reporting paths) before applying here.
_p = 'engine.py'
_s = open(_p).read()

_e_old1 = '''          mAUC, auc_scores = meanAUC(y_test, p_test)
          mMCC, mcc_scores = meanMCC(y_test, p_test)
          mAP, ap_scores = meanAP(y_test, p_test)
          mF1, f1_scores = meanF1(y_test, p_test)
            
          print(">> Mean AUC = {:.4f} \\nAUC = {}".format(mAUC, np.array2string(np.array(auc_scores), precision=4, separator=',')))
          print(">> Mean MCC = {:.4f} \\nMCC = {}".format(mMCC, np.array2string(np.array(mcc_scores), precision=4, separator=',')))
          print(">> Mean AP = {:.4f} \\nAP = {}".format(mAP, np.array2string(np.array(ap_scores), precision=4, separator=',')))
          print(">> Mean F1 = {:.4f} \\nF1 = {}".format(mF1, np.array2string(np.array(f1_scores), precision=4, separator=',')))
          mean_result_list.append(mAUC)
          result_list.append([mAUC,mMCC,mAP,mF1])'''
_e_new1 = '''          mAUC, auc_scores = meanAUC(y_test, p_test)
          mMCC, mcc_scores = meanMCC(y_test, p_test)
          mAP, ap_scores = meanAP(y_test, p_test)
          mF1, f1_scores = meanF1(y_test, p_test)
          mRecall, recall_scores = meanRecall(y_test, p_test)
          mAccuracy, accuracy_scores = meanAccuracy(y_test, p_test)

          print(">> Mean AUC = {:.4f} \\nAUC = {}".format(mAUC, np.array2string(np.array(auc_scores), precision=4, separator=',')))
          print(">> Mean MCC = {:.4f} \\nMCC = {}".format(mMCC, np.array2string(np.array(mcc_scores), precision=4, separator=',')))
          print(">> Mean AP = {:.4f} \\nAP = {}".format(mAP, np.array2string(np.array(ap_scores), precision=4, separator=',')))
          print(">> Mean F1 = {:.4f} \\nF1 = {}".format(mF1, np.array2string(np.array(f1_scores), precision=4, separator=',')))
          print(">> Mean Recall = {:.4f} \\nRecall = {}".format(mRecall, np.array2string(np.array(recall_scores), precision=4, separator=',')))
          print(">> Mean Accuracy = {:.4f} \\nAccuracy = {}".format(mAccuracy, np.array2string(np.array(accuracy_scores), precision=4, separator=',')))
          mean_result_list.append(mAUC)
          result_list.append([mAUC,mMCC,mAP,mF1,mRecall,mAccuracy])'''

_e_old2 = '''        print("=====> Max result:  {} at epoch {}".format(best_rest, best_epoch) )
        print("mAUC, mMCC, mAP, mF1 = {}".format(result_list[best_epoch]))
        with open(output_file, 'a') as writer:
          writer.write("=====> Max result:  {} at epoch {}\\n".format(best_rest, best_epoch))
          writer.write("mAUC, mMCC, mAP, mF1 = {}\\n".format(result_list[best_epoch]))'''
_e_new2 = '''        print("=====> Max result:  {} at epoch {}".format(best_rest, best_epoch) )
        print("mAUC, mMCC, mAP, mF1, mRecall, mAccuracy = {}".format(result_list[best_epoch]))
        with open(output_file, 'a') as writer:
          writer.write("=====> Max result:  {} at epoch {}\\n".format(best_rest, best_epoch))
          writer.write("mAUC, mMCC, mAP, mF1, mRecall, mAccuracy = {}\\n".format(result_list[best_epoch]))'''

_e_old3 = '''    accuracy = []
    mean_auc, mean_mcc, mean_ap, mean_f1 = [],[],[],[]
    metric_dict = {"auc": [], "mcc": [], "ap": [], "f1": []}'''
_e_new3 = '''    accuracy = []
    mean_auc, mean_mcc, mean_ap, mean_f1, mean_recall, mean_accuracy = [],[],[],[],[],[]
    metric_dict = {"auc": [], "mcc": [], "ap": [], "f1": [], "recall": [], "accuracy": []}'''

_e_old4 = '''        mAUC, auc_scores = meanAUC(y_test, p_test)
        mMCC, mcc_scores = meanMCC(y_test, p_test)
        mAP, ap_scores = meanAP(y_test, p_test)
        mF1, f1_scores = meanF1(y_test, p_test)
          
        print(">> Mean AUC = {:.4f} \\nAUC = {}".format(mAUC, np.array2string(np.array(auc_scores), precision=4, separator=',')))
        print(">> Mean MCC = {:.4f} \\nMCC = {}".format(mMCC, np.array2string(np.array(mcc_scores), precision=4, separator=',')))
        print(">> Mean AP = {:.4f} \\nAP = {}".format(mAP, np.array2string(np.array(ap_scores), precision=4, separator=',')))
        print(">> Mean F1 = {:.4f} \\nF1 = {}".format(mF1, np.array2string(np.array(f1_scores), precision=4, separator=',')))
        writer.write("AUC = {}\\n MCC = {}\\nAP = {}\\nF1 = {}\\n".format(np.array2string(np.array(auc_scores), precision=4, separator=','), 
                                                                    np.array2string(np.array(mcc_scores), precision=4, separator=','), 
                                                                    np.array2string(np.array(f1_scores), precision=4, separator=','),
                                                                    np.array2string(np.array(auc_scores), precision=4, separator=',')))
        writer.write("{}: mAUC = {:.4f}, mMCC = {:.4f}, mAP = {:.4f}, mF1 = {:.4f}\\n".format(experiment, mAUC, mMCC, mAP, mF1))'''
_e_new4 = '''        mAUC, auc_scores = meanAUC(y_test, p_test)
        mMCC, mcc_scores = meanMCC(y_test, p_test)
        mAP, ap_scores = meanAP(y_test, p_test)
        mF1, f1_scores = meanF1(y_test, p_test)
        mRecall, recall_scores = meanRecall(y_test, p_test)
        mAccuracy, accuracy_scores = meanAccuracy(y_test, p_test)

        print(">> Mean AUC = {:.4f} \\nAUC = {}".format(mAUC, np.array2string(np.array(auc_scores), precision=4, separator=',')))
        print(">> Mean MCC = {:.4f} \\nMCC = {}".format(mMCC, np.array2string(np.array(mcc_scores), precision=4, separator=',')))
        print(">> Mean AP = {:.4f} \\nAP = {}".format(mAP, np.array2string(np.array(ap_scores), precision=4, separator=',')))
        print(">> Mean F1 = {:.4f} \\nF1 = {}".format(mF1, np.array2string(np.array(f1_scores), precision=4, separator=',')))
        print(">> Mean Recall = {:.4f} \\nRecall = {}".format(mRecall, np.array2string(np.array(recall_scores), precision=4, separator=',')))
        print(">> Mean Accuracy = {:.4f} \\nAccuracy = {}".format(mAccuracy, np.array2string(np.array(accuracy_scores), precision=4, separator=',')))
        writer.write("AUC = {}\\n MCC = {}\\nAP = {}\\nF1 = {}\\nRecall = {}\\nAccuracy = {}\\n".format(np.array2string(np.array(auc_scores), precision=4, separator=','),
                                                                    np.array2string(np.array(mcc_scores), precision=4, separator=','),
                                                                    np.array2string(np.array(ap_scores), precision=4, separator=','),
                                                                    np.array2string(np.array(f1_scores), precision=4, separator=','),
                                                                    np.array2string(np.array(recall_scores), precision=4, separator=','),
                                                                    np.array2string(np.array(accuracy_scores), precision=4, separator=',')))
        writer.write("{}: mAUC = {:.4f}, mMCC = {:.4f}, mAP = {:.4f}, mF1 = {:.4f}, mRecall = {:.4f}, mAccuracy = {:.4f}\\n".format(experiment, mAUC, mMCC, mAP, mF1, mRecall, mAccuracy))'''

_e_old5 = '''        mean_auc.append(mAUC)
        mean_mcc.append(mMCC)
        mean_ap.append(mAP)
        mean_f1.append(mF1)
        metric_dict["auc"].append(auc_scores)
        metric_dict["mcc"].append(mcc_scores)
        metric_dict["ap"].append(ap_scores)
        metric_dict["f1"].append(f1_scores)
        experiment = reader.readline()'''
_e_new5 = '''        mean_auc.append(mAUC)
        mean_mcc.append(mMCC)
        mean_ap.append(mAP)
        mean_f1.append(mF1)
        mean_recall.append(mRecall)
        mean_accuracy.append(mAccuracy)
        metric_dict["auc"].append(auc_scores)
        metric_dict["mcc"].append(mcc_scores)
        metric_dict["ap"].append(ap_scores)
        metric_dict["f1"].append(f1_scores)
        metric_dict["recall"].append(recall_scores)
        metric_dict["accuracy"].append(accuracy_scores)
        experiment = reader.readline()'''

_e_old6 = '''      mean_auc,mean_mcc,mean_ap,mean_f1 = np.array(mean_auc),np.array(mean_mcc),np.array(mean_ap),np.array(mean_f1)
      print(">> All trials: mAUC = {}\\n mMCC = {}\\n mAP = {}\\n mF1 = {}\\n ".format(np.array2string(mean_auc, precision=4, separator=','),
                                                                                   np.array2string(mean_mcc, precision=4, separator=','),
                                                                                   np.array2string(mean_ap, precision=4, separator=','),
                                                                                   np.array2string(mean_f1, precision=4, separator=',')))
      writer.write("All trials: mAUC  = {}\\n mMCC  = {}\\n mAP = {}\\n mF1 = {}\\n ".format(np.array2string(mean_auc, precision=4, separator='\\t'),
                                                                            np.array2string(mean_mcc, precision=4, separator='\\t'),
                                                                            np.array2string(mean_ap, precision=4, separator='\\t'),
                                                                            np.array2string(mean_f1, precision=4, separator='\\t')))
      print(">> Mean / STD over All trials: aAUC = {:.4f}({:.4f}) aMCC = {:.4f}({:.4f}) aAP = {:.4f}({:.4f}) aF1 = {:.4f}({:.4f})  \\n".format(np.mean(mean_auc), np.std(mean_auc),
                                                                                                                                            np.mean(mean_mcc), np.std(mean_mcc),
                                                                                                                                            np.mean(mean_ap), np.std(mean_ap),
                                                                                                                                            np.mean(mean_f1), np.std(mean_f1)))
      writer.write("Mean / STD over All trials: aAUC = {:.4f}({:.4f}) aMCC = {:.4f}({:.4f}) aAP = {:.4f}({:.4f}) aF1 = {:.4f}({:.4f})  \\n".format(np.mean(mean_auc), np.std(mean_auc),
                                                                                                                                                np.mean(mean_mcc), np.std(mean_mcc),
                                                                                                                                                np.mean(mean_ap), np.std(mean_ap),
                                                                                                                                                np.mean(mean_f1), np.std(mean_f1)))
      class_wise_scores = []
      class_wise_scores.extend(get_classwise_mean_std(metric_dict["auc"]))
      class_wise_scores.extend(get_classwise_mean_std(metric_dict["mcc"]))
      class_wise_scores.extend(get_classwise_mean_std(metric_dict["ap"]))
      class_wise_scores.extend(get_classwise_mean_std(metric_dict["f1"]))
      writer.write("AUC/MCC/AP/F1(mean/std): \\n{}\\n".format(class_wise_scores))'''
_e_new6 = '''      mean_auc,mean_mcc,mean_ap,mean_f1,mean_recall,mean_accuracy = np.array(mean_auc),np.array(mean_mcc),np.array(mean_ap),np.array(mean_f1),np.array(mean_recall),np.array(mean_accuracy)
      print(">> All trials: mAUC = {}\\n mMCC = {}\\n mAP = {}\\n mF1 = {}\\n mRecall = {}\\n mAccuracy = {}\\n ".format(np.array2string(mean_auc, precision=4, separator=','),
                                                                                   np.array2string(mean_mcc, precision=4, separator=','),
                                                                                   np.array2string(mean_ap, precision=4, separator=','),
                                                                                   np.array2string(mean_f1, precision=4, separator=','),
                                                                                   np.array2string(mean_recall, precision=4, separator=','),
                                                                                   np.array2string(mean_accuracy, precision=4, separator=',')))
      writer.write("All trials: mAUC  = {}\\n mMCC  = {}\\n mAP = {}\\n mF1 = {}\\n mRecall = {}\\n mAccuracy = {}\\n ".format(np.array2string(mean_auc, precision=4, separator='\\t'),
                                                                            np.array2string(mean_mcc, precision=4, separator='\\t'),
                                                                            np.array2string(mean_ap, precision=4, separator='\\t'),
                                                                            np.array2string(mean_f1, precision=4, separator='\\t'),
                                                                            np.array2string(mean_recall, precision=4, separator='\\t'),
                                                                            np.array2string(mean_accuracy, precision=4, separator='\\t')))
      print(">> Mean / STD over All trials: aAUC = {:.4f}({:.4f}) aMCC = {:.4f}({:.4f}) aAP = {:.4f}({:.4f}) aF1 = {:.4f}({:.4f}) aRecall = {:.4f}({:.4f}) aAccuracy = {:.4f}({:.4f})  \\n".format(np.mean(mean_auc), np.std(mean_auc),
                                                                                                                                            np.mean(mean_mcc), np.std(mean_mcc),
                                                                                                                                            np.mean(mean_ap), np.std(mean_ap),
                                                                                                                                            np.mean(mean_f1), np.std(mean_f1),
                                                                                                                                            np.mean(mean_recall), np.std(mean_recall),
                                                                                                                                            np.mean(mean_accuracy), np.std(mean_accuracy)))
      writer.write("Mean / STD over All trials: aAUC = {:.4f}({:.4f}) aMCC = {:.4f}({:.4f}) aAP = {:.4f}({:.4f}) aF1 = {:.4f}({:.4f}) aRecall = {:.4f}({:.4f}) aAccuracy = {:.4f}({:.4f})  \\n".format(np.mean(mean_auc), np.std(mean_auc),
                                                                                                                                                np.mean(mean_mcc), np.std(mean_mcc),
                                                                                                                                                np.mean(mean_ap), np.std(mean_ap),
                                                                                                                                                np.mean(mean_f1), np.std(mean_f1),
                                                                                                                                                np.mean(mean_recall), np.std(mean_recall),
                                                                                                                                                np.mean(mean_accuracy), np.std(mean_accuracy)))
      class_wise_scores = []
      class_wise_scores.extend(get_classwise_mean_std(metric_dict["auc"]))
      class_wise_scores.extend(get_classwise_mean_std(metric_dict["mcc"]))
      class_wise_scores.extend(get_classwise_mean_std(metric_dict["ap"]))
      class_wise_scores.extend(get_classwise_mean_std(metric_dict["f1"]))
      class_wise_scores.extend(get_classwise_mean_std(metric_dict["recall"]))
      class_wise_scores.extend(get_classwise_mean_std(metric_dict["accuracy"]))
      writer.write("AUC/MCC/AP/F1/Recall/Accuracy(mean/std): \\n{}\\n".format(class_wise_scores))'''

if 'meanRecall(y_test, p_test)' in _s:
    print('engine.py already patched, skipping.')
else:
    for _label, _old, _new in [('block1', _e_old1, _e_new1), ('block2', _e_old2, _e_new2), ('block3', _e_old3, _e_new3),
                                ('block4', _e_old4, _e_new4), ('block5', _e_old5, _e_new5), ('block6', _e_old6, _e_new6)]:
        assert _old in _s, f"engine.py {_label} pattern not found -- repo may have changed upstream"
        _s = _s.replace(_old, _new)
    open(_p, 'w').write(_s)
    print('engine.py patched with recall/accuracy reporting.')

# Patch 9: capture train_loss (now returned by train_one_epoch, patch 6) and write a clean
# per-epoch table (Epoch, train_loss, val_loss, F1, AUC, recall, accuracy) to its own CSV file,
# separate from the verbose text log -- this is what section 5's results cell reads and
# displays as a DataFrame. Rewritten in full after every epoch so it survives an interrupted
# run. Verified locally end-to-end (real 3-epoch run) before applying here.
_p = 'engine.py'
_s = open(_p).read()

_t_old1 = '''      mean_result_list, result_list = [], []
      accuracy = []
      for epoch in range(start_epoch, args.epochs):
        if args.skip_training:
          break
        train_one_epoch(data_loader_train,device, model, criterion, optimizer, epoch)'''
_t_new1 = '''      mean_result_list, result_list = [], []
      accuracy = []
      epoch_table = []
      epoch_table_path = os.path.join(model_path, experiment + "_epoch_table.csv")
      for epoch in range(start_epoch, args.epochs):
        if args.skip_training:
          break
        train_loss = train_one_epoch(data_loader_train,device, model, criterion, optimizer, epoch)'''

_t_old2 = '''          mean_result_list.append(mAUC)
          result_list.append([mAUC,mMCC,mAP,mF1,mRecall,mAccuracy])'''
_t_new2 = '''          mean_result_list.append(mAUC)
          result_list.append([mAUC,mMCC,mAP,mF1,mRecall,mAccuracy])

          epoch_table.append({
            "Epoch": epoch + 1,
            "train_loss": round(train_loss, 4),
            "val_loss": round(val_loss, 4),
            "F1": round(mF1, 4),
            "AUC": round(mAUC, 4),
            "recall": round(mRecall, 4),
            "accuracy": round(mAccuracy, 4),
          })
          with open(epoch_table_path, "w", newline="") as f:
            _w = csv.DictWriter(f, fieldnames=["Epoch", "train_loss", "val_loss", "F1", "AUC", "recall", "accuracy"])
            _w.writeheader()
            _w.writerows(epoch_table)'''

if 'epoch_table_path' in _s:
    print('engine.py epoch-table patch already applied, skipping.')
else:
    assert _t_old1 in _s, "engine.py training-loop header pattern not found -- repo may have changed upstream"
    assert _t_old2 in _s, "engine.py result_list.append pattern not found -- repo may have changed upstream"
    _s = _s.replace(_t_old1, _t_new1).replace(_t_old2, _t_new2)
    open(_p, 'w').write(_s)
    print('engine.py patched with per-epoch results table.')

# Patch 10: save a lightweight (weights-only, no optimizer/scheduler state) checkpoint at
# EVERY epoch, not just when val_loss improves. The original repo only ever keeps a single
# "best so far" checkpoint (overwritten in place each time it improves) -- once training moves
# past an earlier epoch there was no way to go back and load that epoch's specific weights for
# inspection or inference. Opt-in via args.save_every_epoch_checkpoint (set in cell 12); off by
# default leaves the original behavior (and the existing resumable "best" checkpoint) unchanged.
# Weights-only rather than a full checkpoint (state_dict + optimizer + scheduler, like the
# "best" one) to keep per-epoch Drive storage manageable -- verified locally: for swin_tiny the
# full checkpoint was ~332MB vs ~111MB weights-only (AdamW's 2 extra per-parameter moment
# buffers roughly triple the size); the ratio holds for Swin-L too.
_p = 'engine.py'
_s = open(_p).read()

_ck_old = '''        val_loss = evaluate(data_loader_val, device,model, criterion)

        lr_scheduler.step(val_loss)

        if args.test_every_epoch:'''
_ck_new = '''        val_loss = evaluate(data_loader_val, device,model, criterion)

        lr_scheduler.step(val_loss)

        if getattr(args, "save_every_epoch_checkpoint", False):
          torch.save({
            'epoch': epoch + 1,
            'val_loss': val_loss,
            'state_dict': model.state_dict(),
          }, "{}_epoch{}.pth.tar".format(save_model_path, epoch + 1))

        if args.test_every_epoch:'''

if 'save_every_epoch_checkpoint' in _s:
    print('engine.py already patched, skipping.')
else:
    assert _ck_old in _s, "engine.py per-epoch loop pattern not found -- repo may have changed upstream"
    _s = _s.replace(_ck_old, _ck_new)
    open(_p, 'w').write(_s)
    print('engine.py patched with per-epoch checkpoint saving.')

# Patch 11: models.py doesn't support ConvNeXt V2 at all -- and it can't, via timm, in this
# repo: the pinned timm==0.5.4 predates ConvNeXt V2 (added to timm well after this repo's
# 2021-era pins), and upgrading timm here risks breaking the Swin Transformer imports this
# repo also depends on. Loaded via transformers instead (facebook/convnextv2-base-22k-224),
# wrapped with a plain linear head on its pooled output -- the same pattern used for RAD-DINO.
# Verified locally (forward+backward pass, gradients flow into both the head and the encoder)
# before applying here.
_p = 'models.py'
_s = open(_p).read()

_imp_old = "from utils import load_swin_pretrained\n"
_imp_new = ("from utils import load_swin_pretrained\n"
            "from transformers import AutoModel\n"
            "\n"
            "\n"
            "class ConvNextV2Classifier(nn.Module):\n"
            '    """ConvNeXt V2 (facebook/convnextv2-base-22k-224 from Hugging Face) is not\n'
            "    supported by the pinned timm==0.5.4 in this repo (ConvNeXt V2 was only added to\n"
            "    timm in a later version, and upgrading timm here risks breaking the Swin\n"
            "    Transformer imports this repo also depends on) -- loaded via transformers instead,\n"
            "    wrapped with a plain linear head on its pooled output, the same pattern used for\n"
            "    RAD-DINO.\n"
            '    """\n'
            "    def __init__(self, num_classes):\n"
            "        super().__init__()\n"
            '        self.encoder = AutoModel.from_pretrained("facebook/convnextv2-base-22k-224")\n'
            "        self.head = nn.Linear(self.encoder.config.hidden_sizes[-1], num_classes)\n"
            "\n"
            "    def forward(self, x):\n"
            "        out = self.encoder(pixel_values=x)\n"
            '        feat = out.pooler_output if getattr(out, "pooler_output", None) is not None else out.last_hidden_state.mean(dim=[-2, -1])\n'
            "        return self.head(feat)\n")

if 'class ConvNextV2Classifier' in _s:
    print('models.py already patched (patch 11), skipping.')
else:
    assert _imp_old in _s, "models.py import anchor not found -- repo may have changed upstream"
    _s = _s.replace(_imp_old, _imp_new, 1)

_mb_old = ('        elif args.model_name.lower() == "resnet50":\n'
           '            if args.init.lower() =="random":\n'
           "                model = ResNet50(num_classes=args.num_class)\n"
           "        \n"
           "    else:")
_mb_new = ('        elif args.model_name.lower() == "resnet50":\n'
           '            if args.init.lower() =="random":\n'
           "                model = ResNet50(num_classes=args.num_class)\n"
           "\n"
           '        elif args.model_name.lower() == "convnextv2_base":\n'
           "            # Always loads its own pretrained weights from Hugging Face, regardless of\n"
           "            # args.init/args.pretrained_weights -- same reasoning as rad_dino.\n"
           "            model = ConvNextV2Classifier(num_classes=args.num_class)\n"
           "\n"
           "    else:")

if 'model_name.lower() == "convnextv2_base"' in _s:
    print('models.py already patched (convnextv2_base branch), skipping.')
else:
    assert _mb_old in _s, "models.py resnet50/else anchor not found -- repo may have changed upstream"
    _s = _s.replace(_mb_old, _mb_new, 1)
    open(_p, 'w').write(_s)
    print('models.py patched with ConvNextV2Classifier + convnextv2_base model_name.')

# Patch 12: patch 6's bf16 autocast wraps every model's forward pass unconditionally, but
# ConvNeXt V2's depthwise convolutions hit a cuDNN "no engine found" error under bf16 autocast
# on at least some GPU/cuDNN combinations (verified locally: fails under autocast, works fine
# in plain fp32). Makes autocast conditionally toggleable via args.use_autocast (default True,
# so every other notebook's behavior is unchanged) instead of hardcoding it on, and this
# notebook sets it False in cell 12. Verified locally end-to-end (real training run) before
# applying here.
_p = 'trainer.py'
_s = open(_p).read()

if 'use_autocast=True' in _s:
    print('trainer.py already patched (patch 12), skipping.')
else:
    _t1_old = ("def train_one_epoch(data_loader_train, device,model, criterion, optimizer, epoch):\n")
    _t1_new = ("def train_one_epoch(data_loader_train, device,model, criterion, optimizer, epoch, use_autocast=True):\n")
    assert _t1_old in _s, "trainer.py train_one_epoch signature not found -- repo may have changed upstream"
    _s = _s.replace(_t1_old, _t1_new, 1)

    _t2_old = ('    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):\n'
               "      outputs = model(samples)\n"
               "      #outputs = torch.sigmoid(outputs)\n"
               "      loss = criterion(outputs, targets)\n"
               "\n"
               "    optimizer.zero_grad()")
    _t2_new = ('    with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_autocast):\n'
               "      outputs = model(samples)\n"
               "      #outputs = torch.sigmoid(outputs)\n"
               "      loss = criterion(outputs, targets)\n"
               "\n"
               "    optimizer.zero_grad()")
    assert _t2_old in _s, "trainer.py train_one_epoch autocast block not found -- repo may have changed upstream"
    _s = _s.replace(_t2_old, _t2_new, 1)

    _t3_old = ("def evaluate(data_loader_val, device, model, criterion):\n")
    _t3_new = ("def evaluate(data_loader_val, device, model, criterion, use_autocast=True):\n")
    assert _t3_old in _s, "trainer.py evaluate signature not found -- repo may have changed upstream"
    _s = _s.replace(_t3_old, _t3_new, 1)

    _t4_old = ('      with torch.autocast(device_type="cuda", dtype=torch.bfloat16):\n'
               "        outputs = model(samples)\n"
               "        #outputs = torch.sigmoid(outputs)\n"
               "        loss = criterion(outputs, targets)\n"
               "\n"
               "      losses.update(loss.item(), samples.size(0))")
    _t4_new = ('      with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_autocast):\n'
               "        outputs = model(samples)\n"
               "        #outputs = torch.sigmoid(outputs)\n"
               "        loss = criterion(outputs, targets)\n"
               "\n"
               "      losses.update(loss.item(), samples.size(0))")
    assert _t4_old in _s, "trainer.py evaluate autocast block not found -- repo may have changed upstream"
    _s = _s.replace(_t4_old, _t4_new, 1)

    _t5_old = ('      with torch.autocast(device_type="cuda", dtype=torch.bfloat16):\n'
               "        out = model(varInput)\n"
               '      if args.data_set in ["RSNAPneumonia", "COVIDx"]:')
    _t5_new = ('      with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=getattr(args, "use_autocast", True)):\n'
               "        out = model(varInput)\n"
               '      if args.data_set in ["RSNAPneumonia", "COVIDx"]:')
    _count = _s.count(_t5_old)
    assert _count == 2, f"expected 2 occurrences (test_classification + test_model) in trainer.py, found {_count} -- repo may have changed upstream"
    _s = _s.replace(_t5_old, _t5_new)

    open(_p, 'w').write(_s)
    print('trainer.py patched: autocast now conditionally toggleable via use_autocast/args.use_autocast.')

_p2 = 'engine.py'
_s2 = open(_p2).read()

if 'use_autocast=getattr(args' in _s2:
    print('engine.py already patched (patch 12), skipping.')
else:
    _e1_old = "train_loss = train_one_epoch(data_loader_train,device, model, criterion, optimizer, epoch)"
    _e1_new = 'train_loss = train_one_epoch(data_loader_train,device, model, criterion, optimizer, epoch, use_autocast=getattr(args, "use_autocast", True))'
    assert _e1_old in _s2, "engine.py train_one_epoch call site not found -- repo may have changed upstream"
    _s2 = _s2.replace(_e1_old, _e1_new, 1)

    _e2_old = "val_loss = evaluate(data_loader_val, device,model, criterion)"
    _e2_new = 'val_loss = evaluate(data_loader_val, device,model, criterion, use_autocast=getattr(args, "use_autocast", True))'
    assert _e2_old in _s2, "engine.py evaluate call site not found -- repo may have changed upstream"
    _s2 = _s2.replace(_e2_old, _e2_new, 1)

    open(_p2, 'w').write(_s2)
    print('engine.py patched: threads args.use_autocast through both train_one_epoch/evaluate calls.')

In [ ]:
import torch
print('torch', torch.__version__, 'cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0))
from transformers import AutoModel
m = AutoModel.from_pretrained("facebook/convnextv2-base-22k-224")
print('ConvNeXt V2 Base (ImageNet-22k) encoder built OK, params:', sum(p.numel() for p in m.parameters()))
del m

## 3. Mount Google Drive for checkpoint persistence
`/content` is ephemeral — if the runtime disconnects or gets recycled (e.g. Colab Compute Units running out), everything in it is gone, checkpoints included. This mounts Drive and points training's save location there instead, so progress survives even a full runtime loss. Re-running cell 12 after a restart will then automatically resume from the last saved epoch (via `args.resume`) instead of starting over.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE_CKPT_DIR = '/content/drive/MyDrive/convnextv2_chexpert_checkpoints'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print('Checkpoints and results will be saved to:', DRIVE_CKPT_DIR)

## 4. Run finetuning on the CheXpert benchmark
Trains ConvNeXt V2 (ImageNet-22k init) per the `NUM_CLASSES` toggle set in the cell below, reporting AUROC/MCC/AP/F1/Recall/Accuracy on the 5 competition classes each epoch either way. Adjust `BATCH_SIZE`/`EPOCHS`/`PATIENCE` there for your GPU and time budget -- defaults mirror the settings used for the other baseline notebooks' diagnostic runs (generous epoch ceiling, patience effectively disabled, proportionally-scaled warmup/cooldown). Note `args.use_autocast = False` here (see patch 12) -- this run is in plain fp32, not bf16, so expect somewhat slower per-batch time than the other 224px notebooks.

In [ ]:
%cd /content/chexpert_code
import sys, os
sys.path.insert(0, '.')

# main_classification.py's CLI hardcodes CheXpert to all 14 labels (train on 14, eval on 5).
# We want the model head trained on 5 classes only, so we build args the same way the CLI
# would (all its other defaults intact) and call the training engine directly instead.
_old_argv = sys.argv
sys.argv = ['main_classification.py']
from main_classification import get_args_parser
args = get_args_parser()
sys.argv = _old_argv

EPOCHS = 50
# High enough that early stopping won't cut the run short -- mirrors the other baseline
# notebooks' diagnostic runs so we get the full loss/metric history for the over/underfitting check below.
PATIENCE = 50

# 14 = train the head on all 14 CheXpert labels, but only report
#      AUROC/etc. on the 5 competition classes (via test_diseases subsetting below).
# 5  = train the head on the 5 competition classes only (what patch 3 in cell 7 enables).
NUM_CLASSES = 14

# Saves a lightweight (weights-only) checkpoint at every epoch, in addition to the existing
# "best so far" resumable checkpoint -- see patch 10 in cell 7.
SAVE_EVERY_EPOCH_CHECKPOINT = True

# ConvNeXt V2, ImageNet-22k pretrained -- always loads its own Hugging Face weights (added via
# patch 11), there's no random-init variant here since the pretraining is the point of this
# baseline.
args.model_name = "convnextv2_base"
args.init = "convnextv2"
args.pretrained_weights = None
args.input_size = 224
args.img_size = 256
# ConvNeXt V2's depthwise convolutions hit a cuDNN "no engine found" error under bf16 autocast
# on at least some GPU/cuDNN combinations (verified locally). Patch 12 makes this toggleable;
# disabled here for correctness. Expect somewhat slower per-batch time than the other 224px
# notebooks as a result.
args.use_autocast = False
# ConvNeXt V2 Base is roughly similar in size to Swin-B -- same 224px batch size as a
# proven-safe starting point.
BATCH_SIZE = 32

diseases_14 = ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
               'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax',
               'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices']
competition_5 = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Pleural Effusion']
class_indices_5 = [13, 7, 11, 10, 15]  # raw CSV column indices for competition_5, in that order

if NUM_CLASSES == 14:
    diseases = diseases_14
    ds_class_indices = None  # CheXpert dataset falls back to its default (all 14 columns)
    test_diseases = [diseases_14.index(c) for c in competition_5]
else:
    diseases = competition_5
    ds_class_indices = class_indices_5
    test_diseases = None

args.num_class = NUM_CLASSES
args.data_set = "CheXpert"
args.data_dir = "/content/data"
args.train_list = "/content/data/train.csv"
args.val_list = "/content/data/valid.csv"
args.test_list = "/content/data/valid.csv"
args.key = "teacher"
args.batch_size = BATCH_SIZE
args.epochs = EPOCHS
args.patience = PATIENCE
args.opt = "adamw"
args.lr = 5e-5
args.weight_decay = 0.05
args.sched = "cosine"
# main_classification.py's own parser defaults (warmup_epochs=20, cooldown_epochs=10) are
# tuned for 100+ epoch training runs -- at EPOCHS=50 that's 60% of the run spent on
# warmup+cooldown alone, barely any left for the actual cosine decay. Scaled down
# proportionally instead (10% warmup, 10% cooldown, 80% main cosine decay).
args.warmup_epochs = 5
args.cooldown_epochs = 5
args.uncertain_label = "LSR-Ones"
args.test_every_epoch = True
args.save_every_epoch_checkpoint = SAVE_EVERY_EPOCH_CHECKPOINT
# Tagged with model/init/epoch count so this doesn't collide with (or accidentally try to
# resume into) any other experiment's checkpoint directory.
args.exp_name = f"_colab_convnextv2_{NUM_CLASSES}class_{EPOCHS}ep"
args.device = "cuda"

# Checkpoints/results go to Drive (mounted in cell 10) instead of ephemeral /content, and
# resume=True means re-running this cell after a runtime loss picks up from the last saved
# epoch instead of starting over. Set to False for this rerun specifically -- the previous
# run at this same exp_name showed a suspicious high-accuracy/low-recall pattern (likely a
# threshold-calibration issue), so we want a genuinely fresh run, not one that resumes from
# that checkpoint. Flip back to True once you've confirmed this run looks healthy and just
# want crash/disconnect resilience.
args.resume = False

from dataloader import CheXpert, build_transform_classification
from engine import classification_engine

def make_ds(mode, file_path):
    return CheXpert(images_path=args.data_dir, file_path=file_path,
        augment=build_transform_classification(normalize=args.normalization, mode=mode, crop_size=args.input_size, resize=args.img_size),
        uncertain_label=args.uncertain_label, unknown_label=args.unknown_label, class_indices=ds_class_indices)

dataset_train = make_ds("train", args.train_list)
dataset_val = make_ds("valid", args.val_list)
dataset_test = make_ds("test", args.test_list)
print('train/val/test sizes:', len(dataset_train), len(dataset_val), len(dataset_test))
print('model:', args.model_name, '| input_size:', args.input_size, '| batch_size:', args.batch_size, '| use_autocast:', args.use_autocast)

model_path = os.path.join(DRIVE_CKPT_DIR, "Models/Classification", args.data_set)
output_path = os.path.join(DRIVE_CKPT_DIR, "Outputs/Classification", args.data_set)

classification_engine(args, model_path, output_path, diseases, dataset_train, dataset_val, dataset_test, test_diseases=test_diseases)

## 5. Results
Per-epoch AUROC prints during training above. Final summary (mean/std AUC, MCC, AP, F1 across the 5 classes) lands in `Outputs/Classification/CheXpert/*_results.txt`; checkpoints in `Models/Classification/CheXpert/` — both now inside `DRIVE_CKPT_DIR` on your Google Drive (see cell 10), so they survive a runtime loss. If cell 12 gets cut off mid-run, just re-run it — `args.resume=True` picks back up from the last saved epoch automatically.

The next cell shows the clean per-epoch table (Epoch, train_loss, val_loss, F1, AUC, recall, accuracy) as a DataFrame — patch 9 (cell 7) writes this to its own CSV alongside the verbose text log, rewritten after every epoch so it's readable even from a run that got interrupted partway through.

In [ ]:
!find "{DRIVE_CKPT_DIR}/Outputs" -type f
!echo ---
!cat "{DRIVE_CKPT_DIR}/Outputs/Classification/CheXpert"/*_results.txt

In [ ]:
import glob, os

def _resolve_latest_epoch_table():
    try:
        _exact = glob.glob(f"{DRIVE_CKPT_DIR}/Models/Classification/CheXpert/{args.exp_name}/{args.exp_name}_run_*_epoch_table.csv")
        if _exact:
            return sorted(_exact)[-1]
    except NameError:
        pass
    _csvs = glob.glob(f"{DRIVE_CKPT_DIR}/Models/Classification/CheXpert/*/*_epoch_table.csv")
    if not _csvs:
        return None
    return max(_csvs, key=os.path.getmtime)
import pandas as pd

_latest = _resolve_latest_epoch_table()
if _latest is None:
    print("No epoch table found yet -- run cell 12 through at least one epoch first.")
else:
    print("Showing:", _latest)
    display(pd.read_csv(_latest))

## 5b. Loss curves (over/underfitting check) and per-epoch checkpoints
The cell below pulls out just epochs 20, 35, and 50 into their own table. The plot after that reads the same `epoch_table.csv` and plots `train_loss` vs `val_loss` across epochs -- train loss dropping while val loss flattens or rises is the standard overfitting signal; both staying high/flat together points to underfitting.

The second cell lists the per-epoch checkpoint files written by patch 10 (only present if `SAVE_EVERY_EPOCH_CHECKPOINT=True` in cell 12) and shows how to load a specific epoch's weights back into a model for inspection or inference, independent of whichever epoch ended up "best" for resuming training.

In [ ]:
import glob, os

def _resolve_latest_epoch_table():
    try:
        _exact = glob.glob(f"{DRIVE_CKPT_DIR}/Models/Classification/CheXpert/{args.exp_name}/{args.exp_name}_run_*_epoch_table.csv")
        if _exact:
            return sorted(_exact)[-1]
    except NameError:
        pass
    _csvs = glob.glob(f"{DRIVE_CKPT_DIR}/Models/Classification/CheXpert/*/*_epoch_table.csv")
    if not _csvs:
        return None
    return max(_csvs, key=os.path.getmtime)
import pandas as pd

_latest = _resolve_latest_epoch_table()
if _latest is None:
    print("No epoch table found yet -- run cell 12 through at least one epoch first.")
else:
    print("Showing:", _latest)
    _df = pd.read_csv(_latest)

    _epochs_of_interest = [20, 35, 50]
    _subset = _df[_df["Epoch"].isin(_epochs_of_interest)]
    _missing = sorted(set(_epochs_of_interest) - set(_subset["Epoch"]))
    if _missing:
        print(f"Not reached yet: epoch(s) {_missing} (latest epoch so far: {_df['Epoch'].max()})")
    if not _subset.empty:
        display(_subset.set_index("Epoch"))

In [ ]:
import glob, os

def _resolve_latest_epoch_table():
    try:
        _exact = glob.glob(f"{DRIVE_CKPT_DIR}/Models/Classification/CheXpert/{args.exp_name}/{args.exp_name}_run_*_epoch_table.csv")
        if _exact:
            return sorted(_exact)[-1]
    except NameError:
        pass
    _csvs = glob.glob(f"{DRIVE_CKPT_DIR}/Models/Classification/CheXpert/*/*_epoch_table.csv")
    if not _csvs:
        return None
    return max(_csvs, key=os.path.getmtime)
import pandas as pd
import matplotlib.pyplot as plt

_latest = _resolve_latest_epoch_table()
if _latest is None:
    print("No epoch table found yet -- run cell 12 through at least one epoch first.")
else:
    print("Showing:", _latest)
    _df = pd.read_csv(_latest)

    _best_val_loss_epoch = int(_df.loc[_df["val_loss"].idxmin(), "Epoch"])
    _best_val_loss = _df["val_loss"].min()
    _best_auc_epoch = int(_df.loc[_df["AUC"].idxmax(), "Epoch"])
    _best_auc = _df["AUC"].max()
    print(f"Lowest val_loss:  epoch {_best_val_loss_epoch} (val_loss={_best_val_loss:.4f})")
    print(f"Highest val AUC:  epoch {_best_auc_epoch} (AUC={_best_auc:.4f})")

    # Plot 1: train vs val loss, with the lowest-val_loss epoch marked -- the classic
    # overfitting signal is train_loss still falling while val_loss flattens or climbs
    # past this point.
    fig, ax1 = plt.subplots(figsize=(7, 4))
    ax1.plot(_df["Epoch"], _df["train_loss"], marker="o", label="train_loss")
    ax1.plot(_df["Epoch"], _df["val_loss"], marker="o", label="val_loss")
    ax1.axvline(_best_val_loss_epoch, color="gray", linestyle="--", linewidth=1)
    ax1.annotate(f"min val_loss\nepoch {_best_val_loss_epoch}",
                 xy=(_best_val_loss_epoch, _best_val_loss), xytext=(10, 10),
                 textcoords="offset points", fontsize=9, color="gray")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax1.set_title("Train vs Val Loss (over/underfitting check)")
    plt.tight_layout()
    plt.show()

    # Plot 2: val AUC as the primary line (threshold-independent, the actual task-quality
    # metric) with its max marked; F1/Recall/Accuracy shown lighter as secondary context,
    # since they depend on a per-epoch re-picked threshold and are noisier signals.
    fig2, ax2 = plt.subplots(figsize=(7, 4))
    ax2.plot(_df["Epoch"], _df["AUC"], marker="o", linewidth=2.5, color="C0", label="AUC")
    for _col, _c in zip(["F1", "recall", "accuracy"], ["C1", "C2", "C3"]):
        ax2.plot(_df["Epoch"], _df[_col], marker=".", linewidth=1, alpha=0.5, color=_c, label=_col)
    ax2.axvline(_best_auc_epoch, color="gray", linestyle="--", linewidth=1)
    ax2.annotate(f"max AUC\nepoch {_best_auc_epoch}",
                 xy=(_best_auc_epoch, _best_auc), xytext=(10, -15),
                 textcoords="offset points", fontsize=9, color="gray")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Score")
    ax2.legend(loc="lower right")
    ax2.set_title("Val AUC (primary) vs F1 / Sen(Recall) / Accuracy per epoch")
    plt.tight_layout()
    plt.show()

In [ ]:
import glob

_ckpts = sorted(glob.glob(f"{DRIVE_CKPT_DIR}/Models/Classification/CheXpert/*/*_epoch*.pth.tar"))
print("Per-epoch checkpoints found:")
for f in _ckpts:
    print(" ", f)

# Example: load one specific epoch's weights (edit the index/path below).
# import torch
# from models import build_classification_model
# ckpt = torch.load(_ckpts[0], map_location="cpu")
# print("epoch:", ckpt["epoch"], "val_loss:", ckpt["val_loss"])
# model = build_classification_model(args)
# model.load_state_dict(ckpt["state_dict"])

## 6. Publication-style comparison table
Builds a `Method | Acc | Sen | F1 | AUC` table (the format common in chest X-ray classification papers) from every `*_results.txt` file found in `Outputs/Classification/CheXpert/` — one row per experiment you've run (e.g. different `model_name`/`init` configs, or repeats with different settings). "Sen" (Sensitivity) is the same quantity as `Recall` in the per-epoch table above; renamed here to match the convention in most published tables.

Note: with `args.num_trial=1` (the current default), there's only a single run per configuration, so there's no real cross-run standard deviation to report — the `(mean ± std)` style you see in papers comes from averaging several independently-seeded trials. Bump `args.num_trial` in cell 12 (e.g. to 3) and rerun if you want genuine ± error bars later; that costs proportionally more compute.

In [ ]:
import re, glob, os
import pandas as pd

_pattern = re.compile(
    r"aAUC = (-?[\d.]+)\((-?[\d.]+)\)\s*"
    r"aMCC = (-?[\d.]+)\((-?[\d.]+)\)\s*"
    r"aAP = (-?[\d.]+)\((-?[\d.]+)\)\s*"
    r"aF1 = (-?[\d.]+)\((-?[\d.]+)\)\s*"
    r"aRecall = (-?[\d.]+)\((-?[\d.]+)\)\s*"
    r"aAccuracy = (-?[\d.]+)\((-?[\d.]+)\)"
)

def label_from_filename(fname):
    # This notebook only ever trains ConvNeXt V2 variants (no per-class-uncertain-label /
    # class-weighted / co-occurrence-loss experiments here), so just clean up the exp_name
    # itself into a readable label instead of guessing at specific improvement tags.
    base = os.path.basename(fname)
    label = base.replace("_results.txt", "").lstrip("_").replace("_", " ")
    return label

rows = []
for f in sorted(glob.glob(f"{DRIVE_CKPT_DIR}/Outputs/Classification/CheXpert/*_results.txt")):
    text = open(f).read()
    m = _pattern.search(text)
    if not m:
        print("Could not parse (no 'Mean / STD over All trials' line found):", f)
        continue
    auc, auc_std, mcc, mcc_std, ap, ap_std, f1, f1_std, recall, recall_std, acc, acc_std = map(float, m.groups())
    rows.append({
        "Method": label_from_filename(f),
        "Acc (%)": round(acc * 100, 1),
        "Sen (%)": round(recall * 100, 1),
        "F1 (%)": round(f1 * 100, 1),
        "AUC (%)": round(auc * 100, 1),
    })

if not rows:
    print("No results files found yet -- run cell 12 through completion first.")
else:
    df = pd.DataFrame(rows).set_index("Method")
    display(df)